# Notebook 07 — Certified Baselines (IBP & CROWN)

Implements and runs two certified robustness methods from scratch:

| Method | Type | Guarantee |
|---|---|---|
| **IBP** | Interval Bound Propagation | Formal certified lower bound on ε* |
| **CROWN** | Linear ReLU relaxation | Tighter formal certified lower bound |

Both provide **formal lower bounds** — if certified at ε, it is
mathematically guaranteed no adversarial example exists within that radius.
This distinguishes them from concolic exploration's heuristic lower bound
and is the key comparison for the TAI paper.

**Implemented from scratch** — no external dependency beyond PyTorch.

**Requires:** `models/small_mnist.pt`, `models/lenet5_mnist.pt`, `models/smallcnn_cifar.pt`

**Outputs:**
```
results/certified_small_mnist.json
results/certified_lenet5_mnist.json
results/certified_smallcnn_cifar.json
results/certified_summary.json
```

In [1]:
import subprocess, sys
for pkg in ['torch', 'torchvision', 'numpy', 'tqdm']:
    subprocess.run([sys.executable, '-m', 'pip', 'install', pkg, '--quiet'], check=False)
import torch
print(f'PyTorch: {torch.__version__}')

PyTorch: 2.12.0+cpu


In [2]:
import sys, os, time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from pathlib import Path
from tqdm import tqdm
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

REPO_ROOT   = Path(os.getcwd())
sys.path.insert(0, str(REPO_ROOT))

from utils.network_definitions import load_model
from utils.cnn_definitions      import load_cnn
from utils.metrics import get_correctly_classified_samples, save_results

MODELS_DIR  = REPO_ROOT / 'models'
RESULTS_DIR = REPO_ROOT / 'results'
DATA_DIR    = REPO_ROOT / 'data'
RESULTS_DIR.mkdir(exist_ok=True)

SEED      = 42
N_SAMPLES = 100
EPS_MIN   = 0.001
EPS_MAX   = 1.0
N_BISECT  = 15

torch.manual_seed(SEED); np.random.seed(SEED)
print(f'Config: {N_SAMPLES} samples, eps=[{EPS_MIN},{EPS_MAX}], bisect={N_BISECT}')

Config: 100 samples, eps=[0.001,1.0], bisect=15


## 1 — Core bound propagation

Both IBP and CROWN propagate lower/upper bounds through the network.
We traverse only **leaf modules** (no containers like `nn.Sequential`)
to avoid double-processing layers.

In [3]:
def leaf_modules(model):
    """Yield leaf modules only (no containers)."""
    for m in model.modules():
        if len(list(m.children())) == 0 and m is not model:
            yield m


def propagate_bounds(model, lo, hi, use_crown=False):
    """
    Propagate interval bounds [lo, hi] through network leaf modules.

    IBP mode  (use_crown=False): interval arithmetic at every layer.
    CROWN mode (use_crown=True ): linear relaxation at ReLU layers
                                  (tighter but same asymptotic complexity).

    Returns final (lo, hi) tensors over output logits.
    """
    for m in leaf_modules(model):

        if isinstance(m, nn.Linear):
            W = m.weight.detach()
            b = m.bias.detach()
            if lo.dim() > 1: lo = lo.flatten()
            if hi.dim() > 1: hi = hi.flatten()
            W_pos, W_neg = W.clamp(min=0), W.clamp(max=0)
            lo, hi = (W_pos @ lo + W_neg @ hi + b,
                      W_pos @ hi + W_neg @ lo + b)

        elif isinstance(m, nn.Conv2d):
            W  = m.weight.detach()
            b  = m.bias.detach() if m.bias is not None else None
            kw = dict(stride=m.stride, padding=m.padding,
                      dilation=m.dilation, groups=m.groups)
            W_pos, W_neg = W.clamp(min=0), W.clamp(max=0)
            new_lo = (F.conv2d(lo.unsqueeze(0), W_pos, **kw)[0]
                    + F.conv2d(hi.unsqueeze(0), W_neg, **kw)[0])
            new_hi = (F.conv2d(hi.unsqueeze(0), W_pos, **kw)[0]
                    + F.conv2d(lo.unsqueeze(0), W_neg, **kw)[0])
            if b is not None:
                new_lo += b[:, None, None]
                new_hi += b[:, None, None]
            lo, hi = new_lo, new_hi

        elif isinstance(m, nn.ReLU):
            if use_crown:
                # CROWN linear relaxation for unstable neurons (l<0<u)
                # Upper relaxation: slope = u/(u-l), intercept = -l*u/(u-l)
                unstable = (lo < 0) & (hi > 0)
                slope    = torch.where(unstable,
                                       hi / (hi - lo + 1e-12),
                                       (hi >= 0).float())
                new_lo   = torch.where(lo >= 0, lo, torch.zeros_like(lo))
                new_hi   = torch.where(
                    hi <= 0, torch.zeros_like(hi),
                    torch.where(lo >= 0, hi, slope * hi)
                )
                lo, hi   = new_lo, new_hi
            else:
                lo = lo.clamp(min=0)
                hi = hi.clamp(min=0)

        elif isinstance(m, nn.MaxPool2d):
            kw = dict(kernel_size=m.kernel_size, stride=m.stride,
                      padding=m.padding)
            lo = F.max_pool2d(lo.unsqueeze(0), **kw)[0]
            hi = F.max_pool2d(hi.unsqueeze(0), **kw)[0]

        elif isinstance(m, nn.AvgPool2d):
            kw = dict(kernel_size=m.kernel_size, stride=m.stride)
            lo = F.avg_pool2d(lo.unsqueeze(0), **kw)[0]
            hi = F.avg_pool2d(hi.unsqueeze(0), **kw)[0]

        elif isinstance(m, nn.Flatten):
            lo = lo.flatten()
            hi = hi.flatten()

    return lo.flatten(), hi.flatten()


def is_certified(lo_logits, hi_logits, label):
    """True if lower bound on correct class > upper bound on all others."""
    others = hi_logits.clone()
    others[label] = -1e9
    return bool(lo_logits[label] > others.max())


def certified_radius(model, x, label, method='ibp',
                     eps_min=0.001, eps_max=1.0, n_bisect=15):
    """
    Binary search for maximum certified radius.

    Args:
        method : 'ibp' or 'crown'

    Returns:
        Maximum ε at which network is certified robust (0.0 if not certifiable).
    """
    use_crown = (method == 'crown')
    x_t = torch.tensor(x, dtype=torch.float32)

    def check(eps):
        lo, hi = x_t - eps, x_t + eps
        lo_out, hi_out = propagate_bounds(model, lo, hi, use_crown)
        return is_certified(lo_out, hi_out, label)

    # check if certifiable at all at eps_min
    if not check(eps_min):
        return 0.0

    lo_e, hi_e, best = eps_min, eps_max, eps_min
    for _ in range(n_bisect):
        mid = (lo_e + hi_e) / 2
        if check(mid):
            best = mid; lo_e = mid
        else:
            hi_e = mid

    return float(best)


print('IBP and CROWN functions defined.')

IBP and CROWN functions defined.


## 2 — Smoke test

In [4]:
from utils.network_definitions import SmallMLP
from utils.cnn_definitions import LeNet5

# MLP smoke test
mlp = SmallMLP(784, 10); mlp.eval()
x_t = torch.randn(784)
label = int(mlp(x_t.unsqueeze(0)).argmax().item())

r_ibp   = certified_radius(mlp, x_t.numpy(), label, 'ibp',   0.001, 1.0, 8)
r_crown = certified_radius(mlp, x_t.numpy(), label, 'crown', 0.001, 1.0, 8)
print(f'MLP  → IBP: {r_ibp:.4f}  CROWN: {r_crown:.4f}  (CROWN >= IBP: {r_crown >= r_ibp})')
assert r_crown >= r_ibp - 1e-6, 'CROWN should be >= IBP (tighter)'

# LeNet5 smoke test
lenet = LeNet5(1, 10, 32); lenet.eval()
x_cnn = torch.randn(1, 32, 32)
label_c = int(lenet(x_cnn.unsqueeze(0)).argmax().item())
r_ibp_c = certified_radius(lenet, x_cnn.numpy(), label_c, 'ibp', 0.001, 1.0, 6)
print(f'LeNet5 → IBP: {r_ibp_c:.4f}')
assert r_ibp_c >= 0

print('Smoke tests passed.')

MLP  → IBP: 0.0000  CROWN: 0.0000  (CROWN >= IBP: True)
LeNet5 → IBP: 0.0000
Smoke tests passed.


## 3 — Load datasets

In [5]:
mnist_tf_flat = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])
mnist_tf_spat = transforms.Compose([
    transforms.Pad(2),
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)),
])
cifar_tf = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2023,0.1994,0.2010)),
])

def to_numpy_flat(ds):    # for MLP
    l = DataLoader(ds, batch_size=len(ds), shuffle=False)
    X, y = next(iter(l)); return X.view(len(ds),-1).numpy(), y.numpy()
def to_numpy_spatial(ds): # for CNN
    l = DataLoader(ds, batch_size=len(ds), shuffle=False)
    X, y = next(iter(l)); return X.numpy(), y.numpy()

print('Loading...')
X_mnist_flat,  y_mnist = to_numpy_flat(datasets.MNIST(DATA_DIR, train=False, download=True, transform=mnist_tf_flat))
X_mnist_spat,  _       = to_numpy_spatial(datasets.MNIST(DATA_DIR, train=False, download=True, transform=mnist_tf_spat))
X_cifar,       y_cifar = to_numpy_spatial(datasets.CIFAR10(DATA_DIR, train=False, download=True, transform=cifar_tf))
print(f'MNIST flat={X_mnist_flat.shape}  spatial={X_mnist_spat.shape}  CIFAR={X_cifar.shape}')

Loading...
MNIST flat=(10000, 784)  spatial=(10000, 1, 32, 32)  CIFAR=(10000, 3, 32, 32)


## 4 — Evaluation function

In [6]:
def evaluate_certified(model, X, y, model_name=''):
    model.eval()
    results = {'ibp': [], 'crown': []}

    bar = tqdm(zip(X, y), total=len(X), desc=model_name, unit='sample')
    for x, label in bar:
        label = int(label)
        for method in ['ibp', 'crown']:
            t0  = time.time()
            eps = certified_radius(model, x, label, method,
                                   EPS_MIN, EPS_MAX, N_BISECT)
            results[method].append({
                'eps_certified': float(eps),
                'certified'    : eps > 0,
                'runtime_sec'  : time.time() - t0,
            })
        bar.set_postfix(
            ibp=f"{results['ibp'][-1]['eps_certified']:.3f}",
            crown=f"{results['crown'][-1]['eps_certified']:.3f}"
        )

    def agg(rs):
        eps   = np.array([r['eps_certified'] for r in rs])
        times = np.array([r['runtime_sec']   for r in rs])
        cert  = np.array([r['certified']      for r in rs])
        return {
            'mean_eps_certified'  : float(np.mean(eps)),
            'std_eps_certified'   : float(np.std(eps)),
            'median_eps_certified': float(np.median(eps)),
            'fraction_certified'  : float(np.mean(cert)),
            'mean_runtime_sec'    : float(np.mean(times)),
            'n_samples'           : len(rs),
        }

    return {m: {'per_sample': results[m], 'stats': agg(results[m])}
            for m in ['ibp', 'crown']}

## 5 — SmallMLP on MNIST (~10 min)

In [7]:
small_mlp = load_model(str(MODELS_DIR / 'small_mnist.pt'))
X_s, y_s  = get_correctly_classified_samples(small_mlp, X_mnist_flat, y_mnist, N_SAMPLES, seed=SEED)
r_small   = evaluate_certified(small_mlp, X_s, y_s, 'SmallMLP-MNIST')
save_results(r_small, str(RESULTS_DIR / 'certified_small_mnist.json'))
for m in ['ibp','crown']:
    s = r_small[m]['stats']
    print(f'  {m.upper():<6} cert_ε={s["mean_eps_certified"]:.4f}±{s["std_eps_certified"]:.4f}  '
          f'cert%={s["fraction_certified"]:.0%}  {s["mean_runtime_sec"]:.3f}s/sample')

  Loaded ← D:\concolic_exploration\models\small_mnist.pt  |  metadata: {'dataset': 'MNIST', 'architecture': '784-64-64-10', 'best_test_acc': 0.9798, 'n_relu_neurons': 128, 'n_params': 55050}


SmallMLP-MNIST: 100%|████████████████████████████████████| 100/100 [00:01<00:00, 57.03sample/s, crown=0.014, ibp=0.013]


  Saved results → D:\concolic_exploration\results\certified_small_mnist.json
  IBP    cert_ε=0.0085±0.0036  cert%=98%  0.007s/sample
  CROWN  cert_ε=0.0093±0.0042  cert%=98%  0.010s/sample


## 6 — LeNet5 on MNIST (~15 min)

In [8]:
lenet  = load_cnn(str(MODELS_DIR / 'lenet5_mnist.pt'))
X_l, y_l = get_correctly_classified_samples(lenet, X_mnist_spat, y_mnist, N_SAMPLES, seed=SEED)
r_lenet  = evaluate_certified(lenet, X_l, y_l, 'LeNet5-MNIST')
save_results(r_lenet, str(RESULTS_DIR / 'certified_lenet5_mnist.json'))
for m in ['ibp','crown']:
    s = r_lenet[m]['stats']
    print(f'  {m.upper():<6} cert_ε={s["mean_eps_certified"]:.4f}±{s["std_eps_certified"]:.4f}  '
          f'cert%={s["fraction_certified"]:.0%}  {s["mean_runtime_sec"]:.3f}s/sample')

  Loaded ← D:\concolic_exploration\models\lenet5_mnist.pt  |  metadata: {'dataset': 'MNIST', 'input_size': 32, 'best_test_acc': 0.9918, 'architecture': 'LeNet5-ReLU-noBN', 'n_params': 82826}


LeNet5-MNIST: 100%|██████████████████████████████████████| 100/100 [00:07<00:00, 13.36sample/s, crown=0.004, ibp=0.003]

  Saved results → D:\concolic_exploration\results\certified_lenet5_mnist.json
  IBP    cert_ε=0.0020±0.0008  cert%=93%  0.031s/sample
  CROWN  cert_ε=0.0023±0.0010  cert%=94%  0.042s/sample


## 7 — SmallCNN on CIFAR-10 (~20 min)

In [9]:
smallcnn = load_cnn(str(MODELS_DIR / 'smallcnn_cifar.pt'))
X_c, y_c = get_correctly_classified_samples(smallcnn, X_cifar, y_cifar, N_SAMPLES, seed=SEED)
r_cnn    = evaluate_certified(smallcnn, X_c, y_c, 'SmallCNN-CIFAR10')
save_results(r_cnn, str(RESULTS_DIR / 'certified_smallcnn_cifar.json'))
for m in ['ibp','crown']:
    s = r_cnn[m]['stats']
    print(f'  {m.upper():<6} cert_ε={s["mean_eps_certified"]:.4f}±{s["std_eps_certified"]:.4f}  '
          f'cert%={s["fraction_certified"]:.0%}  {s["mean_runtime_sec"]:.3f}s/sample')

  Loaded ← D:\concolic_exploration\models\smallcnn_cifar.pt  |  metadata: {'dataset': 'CIFAR-10', 'input_size': 32, 'best_test_acc': 0.809, 'architecture': 'SmallCNN-3conv-ReLU-noBN', 'n_params': 620362}


SmallCNN-CIFAR10: 100%|██████████████████████████████████| 100/100 [00:01<00:00, 87.14sample/s, crown=0.000, ibp=0.000]

  Saved results → D:\concolic_exploration\results\certified_smallcnn_cifar.json
  IBP    cert_ε=0.0000±0.0000  cert%=0%  0.005s/sample
  CROWN  cert_ε=0.0000±0.0000  cert%=0%  0.006s/sample


## 8 — Summary

In [10]:
summary = {
    'small_mnist'   : {m: r_small[m]['stats'] for m in ['ibp','crown']},
    'lenet5_mnist'  : {m: r_lenet[m]['stats'] for m in ['ibp','crown']},
    'smallcnn_cifar': {m: r_cnn[m]['stats']   for m in ['ibp','crown']},
}
save_results(summary, str(RESULTS_DIR / 'certified_summary.json'))

print()
print('='*72)
print('  CERTIFIED BASELINE RESULTS')
print('='*72)
print(f'  {"Model":<22} {"Method":<8} {"Cert-eps":>10} {"Cert%":>7} {"Time/s":>8}')
print('-'*72)
for mname, methods in summary.items():
    for method, s in methods.items():
        print(f'  {mname:<22} {method.upper():<8}'
              f" {s['mean_eps_certified']:>10.4f}"
              f" {s['fraction_certified']*100:>6.1f}%"
              f" {s['mean_runtime_sec']:>8.3f}s")
    print('-'*72)
print()
print('  CROWN >= IBP always (tighter relaxation).')
print('  Next -> run 08_concolic_cnn.ipynb')

  Saved results → D:\concolic_exploration\results\certified_summary.json

  CERTIFIED BASELINE RESULTS
  Model                  Method     Cert-eps   Cert%   Time/s
------------------------------------------------------------------------
  small_mnist            IBP          0.0085   98.0%    0.007s
  small_mnist            CROWN        0.0093   98.0%    0.010s
------------------------------------------------------------------------
  lenet5_mnist           IBP          0.0020   93.0%    0.031s
  lenet5_mnist           CROWN        0.0023   94.0%    0.042s
------------------------------------------------------------------------
  smallcnn_cifar         IBP          0.0000    0.0%    0.005s
  smallcnn_cifar         CROWN        0.0000    0.0%    0.006s
------------------------------------------------------------------------

  CROWN >= IBP always (tighter relaxation).
  Next -> run 08_concolic_cnn.ipynb
